# FinShield — Exploratory Data Analysis (EDA)

This notebook covers the Phase 1 Exploratory Data Analysis (EDA) of the Kaggle Credit Card Fraud Detection dataset (`data/raw/creditcard.csv`).

## Objectives
1. Understand transaction characteristics (amount, time distribution)
2. Profile class imbalance and fraud rates
3. Analyze correlation patterns among anonymous PCA features
4. Save publication-quality figures to `notebooks/figures/`

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="darkgrid")
RAW_PATH = "../data/raw/creditcard.csv"
FIGURES_DIR = "figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

### Load the Raw Data

In [ ]:
df = pd.read_csv(RAW_PATH)
print(f"Shape of dataset: {df.shape}")
df.head()

### 1. Class Imbalance Profile

Let's look at the split between legitimate and fraudulent transactions.

In [ ]:
counts = df["Class"].value_counts()
fraud_rate = counts[1] / len(df)
print(f"Legitimate count: {counts[0]}")
print(f"Fraudulent count: {counts[1]}")
print(f"Fraud Rate: {fraud_rate:.6%}")

plt.figure(figsize=(6, 4))
sns.barplot(x=counts.index, y=counts.values, palette=["#2E86C1", "#E74C3C"], hue=counts.index, legend=False)
plt.yscale("log")
plt.title("Class Distribution (Log Scale)")
plt.xticks([0, 1], ["Legitimate", "Fraud"])
plt.ylabel("Count (Log Scale)")
plt.savefig(os.path.join(FIGURES_DIR, "class_imbalance.png"), dpi=300)
plt.show()

### 2. Transaction Amount Distribution

Comparing raw amounts to log-transformed amounts.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df["Amount"], bins=50, kde=True, ax=axes[0], color="#2E86C1")
axes[0].set_title("Distribution of Transaction Amount")
axes[0].set_xlabel("Amount (€)")
axes[0].set_yscale("log")

amount_log = np.log1p(df["Amount"])
sns.histplot(amount_log, bins=50, kde=True, ax=axes[1], color="#2ECC71")
axes[1].set_title("Distribution of Log(Amount + 1)")
axes[1].set_xlabel("Log(Amount)")

plt.suptitle("Transaction Amount Distribution Analysis")
plt.savefig(os.path.join(FIGURES_DIR, "amount_distribution.png"), dpi=300)
plt.show()

### 3. Transaction Volume by Hour of Day

In [ ]:
hours = (df["Time"] // 3600) % 24
plt.figure(figsize=(10, 5))
sns.histplot(hours, bins=24, color="#E67E22", kde=True)
plt.title("Transaction Volume by Hour of Day")
plt.xlabel("Hour of Day (0-23)")
plt.ylabel("Transaction Count")
plt.savefig(os.path.join(FIGURES_DIR, "transaction_velocity.png"), dpi=300)
plt.show()

### 4. Correlation with Target (Class)

In [ ]:
plt.figure(figsize=(12, 8))
corr = df.corr()["Class"].drop("Class").sort_values()
sns.barplot(x=corr.values, y=corr.index, palette="coolwarm", hue=corr.index, legend=False)
plt.title("Feature Correlation with Target (Class)")
plt.xlabel("Pearson Correlation Coefficient")
plt.savefig(os.path.join(FIGURES_DIR, "correlation_heatmap.png"), dpi=300)
plt.show()

### 5. Time vs Amount Scatter Plot

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x="Time", y="Amount", hue="Class", palette=["#2E86C1", "#E74C3C"], alpha=0.6, s=15)
plt.title("Time vs Amount Scatter (Colored by Class)")
plt.yscale("log")
plt.savefig(os.path.join(FIGURES_DIR, "time_vs_amount.png"), dpi=300)
plt.show()

### 6. Hourly Fraud Occurrence Count

In [ ]:
df_temp = df.copy()
df_temp["Hour"] = hours
fraud_by_hour = df_temp[df_temp["Class"] == 1].groupby("Hour").size()

plt.figure(figsize=(10, 5))
sns.barplot(x=fraud_by_hour.index, y=fraud_by_hour.values, color="#E74C3C", hue=fraud_by_hour.index, legend=False)
plt.title("Fraudulent Transactions Count by Hour of Day")
plt.xlabel("Hour of Day (0-23)")
plt.ylabel("Number of Fraud Cases")
plt.savefig(os.path.join(FIGURES_DIR, "hourly_fraud_count.png"), dpi=300)
plt.show()